# load dependecies

In [ ]:
## Initialzing and loading required libraries and subfunctions
import numpy as np
import matplotlib.pyplot as plt
import copy
import yasa
from mne.filter import resample
import pynapple as nap
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import normalize
import requests
from io import BytesIO
import sails
from ssqueezepy import cwt
import re
from scipy.stats import entropy
import os, sys

import scipy
from scipy import signal
from scipy.interpolate import griddata
from scipy.signal import correlate
from scipy.stats import pearsonr
from scipy.fft import fft
from scipy.spatial.distance import euclidean
from scipy.signal import spectrogram
from scipy.io import loadmat
import scipy.fft
import scipy.stats
import scipy.io as sio
from scipy.signal import hilbert

import emd as emd
import emd.sift as sift
import emd.spectra as spectra

from neurodsp.sim import sim_combined
from neurodsp.plts import plot_time_series, plot_timefrequency
from neurodsp.utils import create_times
from neurodsp.timefrequency.wavelets import compute_wavelet_transform
from neurodsp.filt import filter_signal

# Load required libraries
import numpy as np
from scipy.io import loadmat
from scipy.signal import hilbert
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns
from neurodsp.filt import filter_signal, filter_signal_fir, design_fir_filter
import emd
import pandas as pd
from sklearn.preprocessing import Normalizer
from tqdm import tqdm
import plotly.express as px
import copy
import umap.umap_ as umap
from scipy.spatial import cKDTree
import pickle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable

## UTILS

for rel in ('src', '../src'):
    p = os.path.abspath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from utils import *
from detect_pt import *

from scipy.io import loadmat
import numpy as np
from neurodsp.filt import filter_signal
import copy
import emd
from scipy.spatial import cKDTree
from tqdm import tqdm

sns.set(style='white', context='notebook')

In [ ]:
path_to_config = '/Users/amir/Desktop/for Abdel/emd_masksift_CA1_config.yml'
config = emd.sift.SiftConfig.from_yaml_file(path_to_config)

# preprocessing

In [ ]:
def extract_cycle_info(imfs, imf_frequencies):

  waveforms = pd.DataFrame()
  all_trials = pd.DataFrame()
  raw_wavelets = []
  all_FPPs = []

  theta_range = [5, 12]
  frequencies = np.arange(15, 141, 1)
  angles=np.linspace(-180,180,19)
  fs = 1000

  for idx, imf in enumerate(imfs):
    cycle_data = get_cycle_data(imf[:, 5], fs)

    amp_thresh = np.percentile(cycle_data['IA'], 25) # higher than 25th percentile of the data
    lo_freq_duration = fs/5  # restrict the analysis to 5-12 Hz
    hi_freq_duration = fs/12

    conditions = ['is_good==1',
                        f'duration_samples<{lo_freq_duration}',
                        f'duration_samples>{hi_freq_duration}',
                        f'max_amp>{amp_thresh}']
    print(len(cycle_data['theta_imf']))
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    
    # Check if any cycles satisfy the conditions
    if all_cycles is None or all_cycles.chain_vect.size == 0:
        print("No cycles satisfy the conditions.")
        return pd.DataFrame(), pd.DataFrame(), []
    
    subset_cycles_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_cycles_df['index'].values

    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycles_inds = arrange_cycle_inds(all_cycles_inds)

    freqs = imf_frequencies[idx]
    sub_theta, theta, supra_theta = tg_split(freqs, theta_range)
    supra_theta_sig = np.sum(imf.T[supra_theta], axis=0)

    # # Corrected Wavelet Transform Computation
    raw_data=sails.wavelet.morlet(supra_theta_sig, freqs=frequencies, sample_rate=fs, ncycles=5,ret_mode='power', normalise='simple')
    raw_wavelets.append(raw_data)
    supraPlot = scipy.stats.zscore(raw_data, axis=1)
    FPP = bin_tf_to_fpp(cycles_inds, supraPlot, bin_count=19)
    all_FPPs.append(FPP)

    # Compute mode frequency for each cycle
    mode_freqs, entropies = compute_mode_frequency_and_entropy(FPP, frequencies, angles)

    all_waveforms, _ = emd.cycles.phase_align(cycle_data['IP'], cycle_data['theta_imf'],
                                                            cycles=all_cycles.iterate(through='subset'), npoints=100)
    all_waveforms = pd.DataFrame(all_waveforms.T)

    waveforms = pd.concat([waveforms, all_waveforms])

    trial = all_cycles.get_metric_dataframe(subset=True)
    trial['mode_freqs'] = mode_freqs
    trial['entropy'] = entropies
    all_trials = pd.concat([all_trials, trial])

  return waveforms, all_trials, all_FPPs

## extract the data

In [ ]:
# =============================================================================
# ========== FPP-derived spectral signatures over the whole dataset ===========
# =============================================================================
import numpy as np
import pandas as pd
import os, re, warnings
import scipy.signal as spsig
from scipy.stats import zscore
import emd

# ------------- Morlet amplitude CWT (same as preprocess v2 path) -------------
def _morlet_amp_spectrogram(x, fs, freqs_hz, w=6.0):
    """
    Morlet CWT amplitude (|complex CWT|), using scipy.signal.cwt + morlet2.
    Returns array [n_freq, n_time].
    """
    freqs_hz = np.asarray(freqs_hz, float)
    scales = (w * fs) / (2.0 * np.pi * freqs_hz)  # s = w*fs/(2π f)
    cwt_mat = spsig.cwt(x, spsig.morlet2, scales, w=w)
    return np.abs(cwt_mat)

# -------------------------- Theta IMF picker (v2-ish) -------------------------
def _choose_theta_imf(imf, fs, imf_centers_hz, theta_band=(5.0, 12.0), prefer_idx=5):
    """
    Prefer a fixed theta IMF index if available; otherwise pick IMF with center
    frequency closest to the theta band center.
    """
    if imf.shape[1] > prefer_idx:
        return prefer_idx
    lo, hi = theta_band
    centers = np.asarray(imf_centers_hz).astype(float)
    if centers.ndim == 1 and centers.size == imf.shape[1]:
        band_center = 0.5*(lo + hi)
        return int(np.argmin(np.abs(centers - band_center)))
    warnings.warn("Unexpected imf_centers_hz shape; defaulting to last IMF.")
    return imf.shape[1] - 1

# --------------------------- Cycle bounds utility -----------------------------
def _cycle_bounds_from_inds(all_cycles_inds):
    """
    Convert per-cycle index arrays -> [start, end] bounds for each cycle.
    """
    bounds = []
    for cyc in all_cycles_inds:
        cyc = np.asarray(cyc)
        cyc = cyc[np.isfinite(cyc)]
        if cyc.size > 1:
            s, e = int(cyc[0]), int(cyc[-1])
            if e > s:
                bounds.append([s, e])
    return np.array(bounds, dtype=int)

# ---------------- FPP-derived spectral signatures for ONE segment -------------
def spectral_signatures_from_fpp_for_segment(
    imf, imf_centers_hz, fs,
    theta_band=(5,12),
    freq_vec=np.arange(15,141,1),
    morlet_w=6.0,
    n_bins=19,
    normalize='zscore',   # 'zscore' or 'none'
):
    """
    Returns:
      sigs_per_cycle : (n_cycles, n_freq) spectral signatures (FPP-based)
      n_cycles       : int, number of valid cycles used
    """
    # 1) theta IMF & cycle selection (same filters you use)
    theta_idx = _choose_theta_imf(imf, fs, imf_centers_hz, theta_band=theta_band, prefer_idx=5)
    cycle_data = get_cycle_data(imf[:, theta_idx], fs=fs)
    if cycle_data is None or 'IA' not in cycle_data:
        return np.zeros((0, len(freq_vec))), 0

    amp_thresh = np.percentile(cycle_data['IA'], 25)
    lo_dur = fs / float(theta_band[0])  # 5 Hz -> shortest period
    hi_dur = fs / float(theta_band[1])  # 12 Hz -> longest period
    conditions = [
        'is_good==1',
        f'duration_samples<{lo_dur}',
        f'duration_samples>{hi_dur}',
        f'max_amp>{amp_thresh}',
    ]
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    if all_cycles is None or getattr(all_cycles, 'chain_vect', np.array([])).size == 0:
        return np.zeros((0, len(freq_vec))), 0

    subset_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_df['index'].values
    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycle_bounds = _cycle_bounds_from_inds(all_cycles_inds)
    if cycle_bounds.size == 0:
        return np.zeros((0, len(freq_vec))), 0

    # 2) supra-theta signal (sum of IMFs > 12 Hz)
    sub_mask, theta_mask, supra_mask = tg_split(imf_centers_hz, theta_band)
    supra_theta_sig = imf[:, supra_mask].sum(axis=1) if np.any(supra_mask) else np.zeros(imf.shape[0])

    # 3) Morlet amplitude TFR (15–140 Hz) and optional normalization
    tf_amp = _morlet_amp_spectrogram(supra_theta_sig, fs, freq_vec, w=morlet_w)  # [n_freq, n_time]
    if normalize == 'zscore':
        tf_use = zscore(tf_amp, axis=1)  # per-freq z across time
    elif normalize == 'none':
        tf_use = tf_amp
    else:
        raise ValueError("normalize must be 'zscore' or 'none'.")

    # 4) Per-cycle FPP -> per-cycle spectral signature (mean across phase bins)
    sigs = []
    for b in cycle_bounds:
        # fpp_single: [1, n_freq, n_bins]  -> squeeze -> [n_freq, n_bins]
        fpp_single = bin_tf_to_fpp(b, tf_use, bin_count=n_bins)
        fpp_single = np.squeeze(fpp_single, axis=0)
        sig_single = np.nanmean(fpp_single, axis=1)  # collapse phase bins
        sigs.append(sig_single)

    if len(sigs) == 0:
        return np.zeros((0, len(freq_vec))), 0
    return np.vstack(sigs), len(sigs)

def extract_data_for_rat_fppsig(rat_id):
    base_path = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition'
    fs = 1000

    cfg = globals().get("RGSconfig", globals().get("config", None))
    if cfg is None:
        raise RuntimeError("Define RGSconfig (preferred) or config before calling this function.")

    all_combined_waveforms = pd.DataFrame()
    all_combined_trials = pd.DataFrame()
    all_phasic_fpps, all_tonic_fpps = [], []
    all_phasic_time_sigs, all_tonic_time_sigs = [], []

    # Support both HC and older CG naming
    preferred_conditions = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD", "HomeCageCG"]
    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    if not conditions:
        print(f"No condition folders found under: {base_path}")
        return None, None

    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d-]+)$')

    for condition_folder in conditions:
        condition_path = os.path.join(base_path, condition_folder)

        # Only keep folders for this rat_id (avoids mismatch spam)
        rat_folders = []
        for f in os.listdir(condition_path):
            full = os.path.join(condition_path, f)
            if not os.path.isdir(full):
                continue
            m = folder_re.match(f)
            if not m:
                continue
            if m.group(1) == str(rat_id):
                rat_folders.append((f, m))

        if not rat_folders:
            continue

        for rat_folder, m in sorted(rat_folders, key=lambda x: x[0]):
            print(f"Processing rat folder: {rat_folder}")
            rat_path = os.path.join(condition_path, rat_folder)

            sd_number = m.group(2)
            condition = m.group(3)
            date_part = m.group(4)

            post_trial_folders = [
                f for f in os.listdir(rat_path)
                if os.path.isdir(os.path.join(rat_path, f)) and re.search(r'Post[-_]Trial\d+', f)
            ]
            post_trial_folders = sorted(post_trial_folders)

            for post_trial_folder in post_trial_folders:
                print(f"Processing post-trial folder: {post_trial_folder}")
                trial_path = os.path.join(rat_path, post_trial_folder)

                lfp_file, state_file = None, None
                for file_name in os.listdir(trial_path):
                    low = file_name.lower()
                    if ('hpc_100' in low) and low.endswith('.mat'):
                        lfp_file = os.path.join(trial_path, file_name)
                    elif ('states' in low) and low.endswith('.mat'):
                        state_file = os.path.join(trial_path, file_name)

                if not lfp_file or not state_file:
                    print(f"Missing LFP or state file in {trial_path}. Skipping...")
                    continue

                trial_number_match = re.search(r'Post[-_]Trial(\d+)', post_trial_folder)
                if not trial_number_match:
                    print(f"Unable to extract trial number from folder name: {post_trial_folder}. Skipping...")
                    continue
                trial_number = int(trial_number_match.group(1))

                try:
                    lfpHPC, hypno, _ = get_data(lfp_file, state_file)
                    try:
                        phasic_interval, tonic_interval, lfp_raw = extract_pt_intervals(lfpHPC, hypno, fs=1000)
                    except ValueError:
                        print(f"No REM sleep found in {post_trial_folder} (Rat {rat_id}, Condition {condition}).")
                        phasic_interval, tonic_interval, lfp_raw = [[], [], []]

                    if not (phasic_interval and tonic_interval):
                        continue

                    tonic_imfs, tonic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, tonic_interval, cfg, return_imfs_freqs=True)
                    phasic_imfs, phasic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, phasic_interval, cfg, return_imfs_freqs=True)

                    phasic_waveforms, phasic_trials, phasic_fpps = extract_cycle_info(phasic_imfs, phasic_freqs)
                    tonic_waveforms, tonic_trials, tonic_fpps = extract_cycle_info(tonic_imfs, tonic_freqs)
                    all_phasic_fpps.extend(phasic_fpps)
                    all_tonic_fpps.extend(tonic_fpps)

                    for seg_imf, seg_freqs in zip(phasic_imfs, phasic_freqs):
                        sigs, _ = spectral_signatures_from_fpp_for_segment(
                            seg_imf, seg_freqs, fs,
                            theta_band=(5, 12),
                            freq_vec=np.arange(15, 141, 1),
                            morlet_w=6.0,
                            n_bins=19,
                            normalize='zscore'
                        )
                        all_phasic_time_sigs.append(sigs)

                    for seg_imf, seg_freqs in zip(tonic_imfs, tonic_freqs):
                        sigs, _ = spectral_signatures_from_fpp_for_segment(
                            seg_imf, seg_freqs, fs,
                            theta_band=(5, 12),
                            freq_vec=np.arange(15, 141, 1),
                            morlet_w=6.0,
                            n_bins=19,
                            normalize='zscore'
                        )
                        all_tonic_time_sigs.append(sigs)

                    for df in [phasic_waveforms, phasic_trials]:
                        df['rat_id'] = rat_id
                        df['condition'] = condition
                        df['trial'] = trial_number
                        df['cycle_type'] = 'phasic'
                        df['SD'] = sd_number
                        df['date'] = date_part

                    for df in [tonic_waveforms, tonic_trials]:
                        df['rat_id'] = rat_id
                        df['condition'] = condition
                        df['trial'] = trial_number
                        df['cycle_type'] = 'tonic'
                        df['SD'] = sd_number
                        df['date'] = date_part

                    all_combined_waveforms = pd.concat(
                        [all_combined_waveforms, phasic_waveforms, tonic_waveforms], ignore_index=True)
                    all_combined_trials = pd.concat(
                        [all_combined_trials, phasic_trials, tonic_trials], ignore_index=True)

                except FileNotFoundError:
                    print(f"Data not found in {trial_path}. Skipping...")

    if all_combined_waveforms.empty:
        print(f"No data extracted for Rat {rat_id}.")
        return None, None

    return (all_combined_waveforms, all_combined_trials,
            all_phasic_fpps, all_tonic_fpps,
            all_phasic_time_sigs, all_tonic_time_sigs)

In [ ]:
import pickle
import pandas as pd
from pathlib import Path

# ===== CONFIG =====
rat_ids  = [1, 2, 3, 4, 6, 7, 8, 9]
out_root = Path("./fppsig_rgs_bwtfilter_linearFreq")
out_root.mkdir(parents=True, exist_ok=True)

# toggle between saving separately or as one pickle
save_all_together = False   # ← True = all_objects.pkl, False = waveforms_df + trials_df + lists.pkl

def _save_df(df: pd.DataFrame, path_no_ext: Path) -> str:
    """Save DataFrame as Parquet if possible; else CSV. Return saved path."""
    try:
        p = path_no_ext.with_suffix(".parquet")
        df.to_parquet(str(p), index=False)
        return str(p)
    except Exception:
        p = path_no_ext.with_suffix(".csv")
        df.to_csv(str(p), index=False)
        return str(p)

manifest_rows = []

for rid in rat_ids:
    print(f"\n=== Rat {rid} ===")
    res = extract_data_for_rat_fppsig(str(rid))
    if not isinstance(res, tuple) or len(res) != 6:
        print(f"  Skipping Rat {rid}: unexpected return value.")
        continue

    (waveforms_df,
     trials_df,
     all_phasic_FPPs,
     all_tonic_FPPs,
     phasic_time_signatures,
     tonic_time_signatures) = res

    if waveforms_df is None or trials_df is None:
        print(f"  Skipping Rat {rid}: no data.")
        continue

    rat_dir = out_root / f"Rat{rid}"
    rat_dir.mkdir(parents=True, exist_ok=True)

    if save_all_together:
        # ----- one combined file -----
        all_path = rat_dir / "all_objects.pkl"
        with open(all_path, "wb") as f:
            pickle.dump(
                dict(
                    waveforms_df=waveforms_df,
                    trials_df=trials_df,
                    all_phasic_FPPs=all_phasic_FPPs,
                    all_tonic_FPPs=all_tonic_FPPs,
                    phasic_time_signatures=phasic_time_signatures,
                    tonic_time_signatures=tonic_time_signatures
                ),
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )
        wf_path = tr_path = lists_path = None
        print(f"  Saved everything → {all_path}")

    else:
        # ----- separate files -----
        wf_path = _save_df(waveforms_df, rat_dir / "waveforms_df")
        tr_path = _save_df(trials_df,   rat_dir / "trials_df")

        lists_path = rat_dir / "lists.pkl"
        with open(lists_path, "wb") as f:
            pickle.dump(
                dict(
                    all_phasic_FPPs=all_phasic_FPPs,
                    all_tonic_FPPs=all_tonic_FPPs,
                    phasic_time_signatures=phasic_time_signatures,
                    tonic_time_signatures=tonic_time_signatures
                ),
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )
        print(f"  Saved separate files → {rat_dir}")

    # --- manifest info ---
    n_phasic_segments = len(phasic_time_signatures)
    n_tonic_segments  = len(tonic_time_signatures)
    n_phasic_cycles   = int(sum((arr.shape[0] for arr in phasic_time_signatures if hasattr(arr, "shape")), 0))
    n_tonic_cycles    = int(sum((arr.shape[0] for arr in tonic_time_signatures  if hasattr(arr, "shape")), 0))

    manifest_rows.append({
        "rat_id": rid,
        "n_phasic_segments": n_phasic_segments,
        "n_tonic_segments": n_tonic_segments,
        "n_phasic_cycles_total": n_phasic_cycles,
        "n_tonic_cycles_total": n_tonic_cycles,
        "waveforms_df_path": wf_path,
        "trials_df_path": tr_path,
        "lists_pickle_path": str(lists_path) if not save_all_together else None,
        "all_objects_pickle_path": str(all_path) if save_all_together else None
    })

# save manifest
manifest_df = pd.DataFrame(manifest_rows).sort_values("rat_id").reset_index(drop=True)
manifest_path = out_root / "manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print(f"\nManifest saved to: {manifest_path}")
print(manifest_df)

## laod all rats

In [ ]:
import pickle
import pandas as pd
from pathlib import Path

def load_rat_data(rid, out_root=Path("./fppsig_rgs_bwtfilter_linearFreq")):
    """Load all data objects for a single rat."""
    rat_dir = out_root / f"Rat{rid}"

    # Option 1: combined pickle
    all_pkl = rat_dir / "all_objects.pkl"
    if all_pkl.exists():
        with open(all_pkl, "rb") as f:
            return pickle.load(f)

    # Option 2: separate files
    def _load_df(name):
        for ext in [".parquet", ".csv"]:
            p = rat_dir / f"{name}{ext}"
            if p.exists():
                return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
        return None

    waveforms_df = _load_df("waveforms_df")
    trials_df = _load_df("trials_df")

    lists_pkl = rat_dir / "lists.pkl"
    if lists_pkl.exists():
        with open(lists_pkl, "rb") as f:
            lists = pickle.load(f)
    else:
        lists = {}

    return dict(waveforms_df=waveforms_df, trials_df=trials_df, **lists)


def load_all_rats(out_root=Path("./fppsig_rgs_bwtfilter_linearFreq")):
    """Load all rats listed in manifest.csv into a dict."""
    manifest_path = out_root / "manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found at {manifest_path}")

    manifest = pd.read_csv(manifest_path)
    rat_ids = manifest["rat_id"].tolist()

    all_data = {}
    for rid in rat_ids:
        print(f"Loading Rat {rid} ...", end=" ")
        try:
            all_data[rid] = load_rat_data(rid, out_root)
            print("✅ done")
        except Exception as e:
            print(f"⚠️ failed ({e})")

    return all_data

In [ ]:
all_rats = load_all_rats()

# extract tSCs - all rats

In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from sklearn.decomposition import PCA, FastICA

try:
    from scipy.ndimage import gaussian_filter1d
    _HAS_SCI = True
except Exception:
    _HAS_SCI = False


def load_rat_data(rid, out_root=Path("./fppsig_v3_outputs")):
    """Load all data objects for a single rat."""
    rat_dir = out_root / f"Rat{rid}"
    
    # Option 1: combined pickle
    all_pkl = rat_dir / "all_objects.pkl"
    if all_pkl.exists():
        with open(all_pkl, "rb") as f:
            return pickle.load(f)
    
    # Option 2: separate files
    def _load_df(name):
        for ext in [".parquet", ".csv"]:
            p = rat_dir / f"{name}{ext}"
            if p.exists():
                return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
        return None
    
    waveforms_df = _load_df("waveforms_df")
    trials_df = _load_df("trials_df")
    
    lists_pkl = rat_dir / "lists.pkl"
    if lists_pkl.exists():
        with open(lists_pkl, "rb") as f:
            lists = pickle.load(f)
    else:
        lists = {}
    
    return dict(waveforms_df=waveforms_df, trials_df=trials_df, **lists)


def load_all_rats(out_root=Path("./fppsig_v3_outputs")):
    """Load all rats listed in manifest.csv into a dict."""
    manifest_path = out_root / "manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found at {manifest_path}")
    
    manifest = pd.read_csv(manifest_path)
    rat_ids = manifest["rat_id"].tolist()
    
    all_data = {}
    for rid in rat_ids:
        print(f"Loading Rat {rid} ...", end=" ")
        try:
            all_data[rid] = load_rat_data(rid, out_root)
            print("✅ done")
        except Exception as e:
            print(f"⚠️ failed ({e})")
    
    return all_data


def run_multi_rat_tSC_pipeline(all_rats_data, 
                                n_pca=5, 
                                freq_vec=np.arange(15, 141, 1),
                                zscore_features=True, 
                                mad_k=2.0,
                                weighted_alpha=0.4,
                                ignore_edge_bins=1):
    """
    Run tSC pipeline on combined data from all rats.
    
    Parameters:
    -----------
    all_rats_data : dict
        Dictionary with rat_id as keys, each containing:
        - phasic_time_signatures: list of arrays
        - tonic_time_signatures: list of arrays
    
    Returns:
    --------
    cycles_df : pd.DataFrame
        DataFrame with all cycles from all rats, including rat_id and all tSC metrics
    tSC_results : dict
        Dictionary containing PCA/ICA components and other analysis results
    """
    
    # ============ 1) Combine data from all rats ============
    print("Combining data from all rats...")
    
    all_phasic_sigs = []
    all_tonic_sigs = []
    rat_metadata = []  # track which rat each signature belongs to
    
    for rat_id, data in all_rats_data.items():
        phasic_sigs = data.get('phasic_time_signatures', [])
        tonic_sigs = data.get('tonic_time_signatures', [])
        
        # Track rat_id for each segment
        for seg in phasic_sigs:
            if isinstance(seg, np.ndarray) and seg.size > 0:
                all_phasic_sigs.append(seg)
                rat_metadata.append({'rat_id': rat_id, 'cycle_type': 'phasic'})
        
        for seg in tonic_sigs:
            if isinstance(seg, np.ndarray) and seg.size > 0:
                all_tonic_sigs.append(seg)
                rat_metadata.append({'rat_id': rat_id, 'cycle_type': 'tonic'})
    
    print(f"  Total phasic segments: {len(all_phasic_sigs)}")
    print(f"  Total tonic segments: {len(all_tonic_sigs)}")
    
    # ============ 2) Flatten all signatures ============
    def _flatten_sig_list(sig_list, metadata_list, label):
        rows, meta = [], []
        seg_idx = 0
        
        for sig_group in sig_list:
            # Find corresponding metadata
            rat_meta = [m for m in metadata_list if m['cycle_type'] == label][seg_idx] if seg_idx < len([m for m in metadata_list if m['cycle_type'] == label]) else {}
            
            if not isinstance(sig_group, np.ndarray) or sig_group.size == 0:
                continue
            
            mask = np.isfinite(sig_group).all(axis=1)
            Xi = sig_group[mask]
            
            if Xi.size == 0:
                continue
            
            rows.append(Xi)
            for j in range(Xi.shape[0]):
                meta.append({
                    "interval_idx": seg_idx,
                    "cycle_idx_in_interval": j,
                    "cycle_type": label,
                    "rat_id": rat_meta.get('rat_id', None)
                })
            
            seg_idx += 1
        
        if len(rows) == 0:
            return np.zeros((0, len(freq_vec))), []
        
        return np.vstack(rows), meta
    
    # Separate metadata by cycle type
    phasic_meta = [m for m in rat_metadata if m['cycle_type'] == 'phasic']
    tonic_meta = [m for m in rat_metadata if m['cycle_type'] == 'tonic']
    
    X_phasic, meta_phasic = _flatten_sig_list(all_phasic_sigs, phasic_meta, "phasic")
    X_tonic, meta_tonic = _flatten_sig_list(all_tonic_sigs, tonic_meta, "tonic")
    
    X = np.vstack([X_phasic, X_tonic])
    meta = meta_phasic + meta_tonic
    
    if X.shape[0] == 0:
        raise RuntimeError("No valid cycles to analyze.")
    
    print(f"Total cycles combined: {X.shape[0]}")
    print(f"  Phasic: {X_phasic.shape[0]}")
    print(f"  Tonic: {X_tonic.shape[0]}")
    
    # ============ 3) Feature z-score ============
    if zscore_features:
        mu = X.mean(axis=0, keepdims=True)
        sd = X.std(axis=0, ddof=1, keepdims=True) + 1e-12
        Xz = (X - mu) / sd
    else:
        Xz = X
    
    # ============ 4) Smoothing helper ============
    def _smooth_rows(mat, sigma_hz, mode="reflect"):
        if _HAS_SCI:
            return gaussian_filter1d(mat, sigma=float(sigma_hz), axis=1, mode=mode)
        
        sigma = float(sigma_hz)
        rad = int(np.ceil(4 * sigma))
        kx = np.arange(-rad, rad + 1)
        ker = np.exp(-(kx**2) / (2 * sigma**2))
        ker /= ker.sum()
        pad = rad
        out = np.empty_like(mat)
        
        for i in range(mat.shape[0]):
            row = mat[i]
            if mode == "reflect":
                row_pad = np.r_[row[pad:0:-1], row, row[-2:-pad-2:-1]]
            elif mode == "constant":
                row_pad = np.r_[np.zeros(pad), row, np.zeros(pad)]
            else:
                row_pad = np.r_[row[pad:0:-1], row, row[-2:-pad-2:-1]]
            out[i] = np.convolve(row_pad, ker, mode="same")[pad:-pad]
        
        return out
    
    # ============ 5) Create smoothed versions ============
    Xz_smooth5_ref = _smooth_rows(Xz, 5.0, mode="reflect")
    Xz_smooth10_ref = _smooth_rows(Xz, 10.0, mode="reflect")
    Xz_smooth5_pad = _smooth_rows(Xz, 5.0, mode="constant")
    Xz_smooth10_pad = _smooth_rows(Xz, 10.0, mode="constant")
    
    # ============ 6) PCA → ICA ============
    print("Running PCA...")
    pca = PCA(n_components=n_pca, random_state=42)
    Z = pca.fit_transform(Xz)
    
    print("Running ICA...")
    ica = FastICA(n_components=n_pca, random_state=42, max_iter=1000, tol=1e-3)
    S_latent = ica.fit_transform(Z)
    W_freq = ica.components_ @ pca.components_
    
    # ============ 7) Sign fix ============
    mean_src = S_latent.mean(axis=0, keepdims=True)
    signs = np.sign(mean_src)
    signs[signs == 0] = 1
    S_latent *= signs
    W_freq *= signs.T
    
    # ============ 8) Compute strengths ============
    strengths = Xz @ W_freq.T
    abs_strengths = np.abs(strengths)
    
    # ============ 9) Thresholds ============
    def _mad_threshold(x, k=2.0):
        med = np.median(x)
        mad = np.median(np.abs(x - med)) + 1e-12
        return med + k * (mad / 0.6745)
    
    thr_per_comp = np.array([_mad_threshold(abs_strengths[:, k], k=mad_k) 
                              for k in range(n_pca)])
    
    labels_0based = np.argmax(abs_strengths, axis=1)
    
    # ============ 10) Winner-strong mask ============
    strong_mask = np.zeros_like(labels_0based, dtype=bool)
    for k in range(n_pca):
        strong_mask |= (labels_0based == k) & (abs_strengths[:, k] >= thr_per_comp[k])
    
    # ============ 11) Paper-style per-component strong ============
    strong_mask_matrix = abs_strengths >= thr_per_comp[None, :]
    tsc_n_strong = strong_mask_matrix.sum(axis=1)
    tsc_any_strong = tsc_n_strong > 0
    tsc_strong_components = [list(np.nonzero(row)[0] + 1) for row in strong_mask_matrix]
    tsc_strong_components_str = [",".join(map(str, lst)) if len(lst) else "" 
                                  for lst in tsc_strong_components]
    tsc_exclusive_label = np.where(tsc_n_strong == 1,
                                    (np.argmax(strong_mask_matrix, axis=1) + 1),
                                    np.nan)
    
    # ============ 12) Mode extraction helper ============
    def _mode_from_mat(mat):
        L, R = ignore_edge_bins, (mat.shape[1] - ignore_edge_bins)
        if R <= L:
            idx = np.argmax(mat, axis=1)
        else:
            idx = np.argmax(mat[:, L:R], axis=1) + L
        return freq_vec[idx]
    
    # ============ 13) Extract modes ============
    mode_freq_hz_featZ = _mode_from_mat(Xz)
    mode_freq_hz_featZ_smooth = _mode_from_mat(Xz_smooth5_ref)
    mode_freq_hz_featZ_smooth10 = _mode_from_mat(Xz_smooth10_ref)
    mode_freq_hz_featZ_smooth5_pad = _mode_from_mat(Xz_smooth5_pad)
    mode_freq_hz_featZ_smooth10_pad = _mode_from_mat(Xz_smooth10_pad)
    
    # ============ 14) Strong modes ============
    def _mode_strong(mat):
        modes = np.full(mat.shape[0], np.nan)
        modes[strong_mask] = _mode_from_mat(mat[strong_mask])
        return modes
    
    mode_freq_hz_featZ_strong = _mode_strong(Xz)
    mode_freq_hz_featZ_smooth_strong = _mode_strong(Xz_smooth5_ref)
    mode_freq_hz_featZ_smooth10_strong = _mode_strong(Xz_smooth10_ref)
    
    # ============ 15) Weighted modes ============
    def mode_from_feature_z_weighted(Xz_like, W_freq, labels_0based, freq_vec,
                                      avoid_edge_bins=1, alpha=None):
        n_cycles, n_freq = Xz_like.shape
        lo = avoid_edge_bins
        hi = n_freq - avoid_edge_bins
        use_slice = slice(lo, hi) if hi > lo else slice(0, n_freq)
        modes = np.empty(n_cycles, dtype=float)
        
        for i in range(n_cycles):
            k = int(labels_0based[i])
            w = np.abs(W_freq[k]).copy()
            
            if alpha is not None:
                thr = alpha * np.max(w)
                m = (w >= thr)
                if m.sum() >= 3:
                    w[~m] = 0.0
            
            y = Xz_like[i] * w
            yseg = y[use_slice]
            idx_rel = int(np.argmax(yseg))
            idx = (lo + idx_rel) if hi > lo else idx_rel
            modes[i] = freq_vec[idx]
        
        return modes
    
    mode_freq_hz_featZ_weighted = mode_from_feature_z_weighted(
        Xz, W_freq, labels_0based, freq_vec,
        avoid_edge_bins=ignore_edge_bins, alpha=weighted_alpha
    )
    
    # ============ 16) Within-cycle-Z + weights ============
    eps = 1e-12
    
    def tsc_weighted_mode_freq(X, W_freq, labels_0based, freq_vec,
                                avoid_edge_bins=1, alpha=0.4):
        n_cycles, n_freq = X.shape
        modes = np.empty(n_cycles, dtype=float)
        lo = avoid_edge_bins
        hi = n_freq - avoid_edge_bins
        use_slice = slice(lo, hi) if hi > lo else slice(0, n_freq)
        
        for i in range(n_cycles):
            k = int(labels_0based[i])
            x = X[i]
            xz = (x - x.mean()) / (x.std(ddof=1) + eps)
            w = np.abs(W_freq[k]).copy()
            
            if alpha is not None:
                thr = alpha * np.max(w)
                mask = w >= thr
                if mask.sum() >= 3:
                    w[~mask] = 0.0
            
            y = xz * w
            yseg = y[use_slice]
            idx_rel = int(np.argmax(yseg))
            idx = (lo + idx_rel) if hi > lo else idx_rel
            modes[i] = freq_vec[idx]
        
        return modes
    
    mode_freq_hz_proj = tsc_weighted_mode_freq(
        X, W_freq, labels_0based, freq_vec,
        avoid_edge_bins=ignore_edge_bins, alpha=weighted_alpha
    )
    
    # ============ 17) Package results ============
    cycles_df = pd.DataFrame(meta)
    cycles_df["tSC_label"] = labels_0based + 1
    cycles_df["mode_freq_hz_featZ"] = mode_freq_hz_featZ
    cycles_df["mode_freq_hz_featZ_smooth"] = mode_freq_hz_featZ_smooth
    cycles_df["mode_freq_hz_featZ_smooth10"] = mode_freq_hz_featZ_smooth10
    cycles_df["mode_freq_hz_featZ_smooth5_pad"] = mode_freq_hz_featZ_smooth5_pad
    cycles_df["mode_freq_hz_featZ_smooth10_pad"] = mode_freq_hz_featZ_smooth10_pad
    cycles_df["mode_freq_hz_featZ_strong"] = mode_freq_hz_featZ_strong
    cycles_df["mode_freq_hz_featZ_smooth_strong"] = mode_freq_hz_featZ_smooth_strong
    cycles_df["mode_freq_hz_featZ_smooth10_strong"] = mode_freq_hz_featZ_smooth10_strong
    cycles_df["mode_freq_hz_featZw"] = mode_freq_hz_featZ_weighted
    cycles_df["mode_freq_hz_proj"] = mode_freq_hz_proj
    
    # Per-component strengths/flags
    for k in range(n_pca):
        cycles_df[f"tSC{k+1}_strength"] = strengths[:, k]
        cycles_df[f"tSC{k+1}_strong"] = (abs_strengths[:, k] >= thr_per_comp[k])
    
    # Paper-style summaries
    cycles_df["tSC_any_strong"] = tsc_any_strong
    cycles_df["tSC_n_strong"] = tsc_n_strong
    cycles_df["tSC_strong_components"] = tsc_strong_components
    cycles_df["tSC_strong_components_str"] = tsc_strong_components_str
    cycles_df["tSC_winner_strong"] = strong_mask
    cycles_df["tSC_exclusive_label"] = tsc_exclusive_label
    
    tSC_results = {
        "freq_vec": freq_vec,
        "pca": pca,
        "ica": ica,
        "weights_freq": W_freq,
        "strengths": strengths,
        "latent_sources_S": S_latent,
        "thresholds_abs_strength": thr_per_comp,
        "X_cycles": X,
        "X_cycles_featZ": Xz,
        "X_cycles_featZ_smooth": Xz_smooth5_ref,
        "X_cycles_featZ_smooth10": Xz_smooth10_ref,
        "X_cycles_featZ_smooth5_pad": Xz_smooth5_pad,
        "X_cycles_featZ_smooth10_pad": Xz_smooth10_pad,
        "strong_mask_matrix": strong_mask_matrix,
        "winner_strong_mask": strong_mask,
        "meta": meta
    }
    
    # ============ 18) Print summary ============
    print("\n=== Multi-Rat PCA/ICA Complete ===")
    print(f"Total cycles: {X.shape[0]}")
    print(f"Mode — featZ (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ']):.2f} Hz")
    print(f"Mode — featZ strong (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ_strong']):.2f} Hz")
    print(f"Mode — featZ 5Hz reflect strong (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ_smooth_strong']):.2f} Hz")
    print(f"Mode — featZ 10Hz reflect strong (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ_smooth10_strong']):.2f} Hz")
    
    # Per-rat summary
    print("\nPer-rat cycle counts:")
    rat_counts = cycles_df.groupby('rat_id').size()
    for rat_id, count in rat_counts.items():
        print(f"  Rat {rat_id}: {count} cycles")
    
    return cycles_df, tSC_results


In [ ]:
# Run multi-rat tSC pipeline
print("\nRunning multi-rat tSC pipeline...")
cycles_df, tSC_results = run_multi_rat_tSC_pipeline(
        all_rats,
        n_pca=5,
        freq_vec=np.arange(15, 141, 1),
        zscore_features=True,
        mad_k=2.0,
        weighted_alpha=0.4,
        ignore_edge_bins=1
    )
    
# Save results
output_dir = Path("./multi_rat_tSC_results")
output_dir.mkdir(parents=True, exist_ok=True)
    
# Save cycles dataframe
cycles_df.to_csv(output_dir / "cycles_df_all_rats.csv", index=False)
print(f"\nSaved cycles_df to {output_dir / 'cycles_df_all_rats.csv'}")
    
# Save tSC results as pickle
with open(output_dir / "tSC_results_all_rats.pkl", "wb") as f:
    pickle.dump(tSC_results, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"Saved tSC_results to {output_dir / 'tSC_results_all_rats.pkl'}")
    
print("\n✅ Analysis complete!")

In [ ]:
# ============================================================
# E) SMALL UTILITIES
# ============================================================

def plot_tSC_weights(weights, freqs):
    plt.figure(figsize=(12, 6))
    
    # Color palette avoiding green - using blues, reds, purples, oranges, browns
    color_palette = [
        '#1f77b4',  # blue
        '#ff7f0e',  # orange  
        '#9467bd',  # purple
        '#8c564b',  # brown
        '#e377c2',  # pink
        '#7f7f7f',  # gray
        '#bcbd22',  # olive (not green)
        '#17becf',  # cyan
        '#d62728',  # red
        '#ff9896'   # light red
    ]
    
    # Extend palette if needed
    while len(color_palette) < weights.shape[0]:
        color_palette.extend(color_palette)
    
    for k in range(weights.shape[0]):
        w = weights[k]
        pk = freqs[np.argmax(np.abs(w))]
        color = color_palette[k % len(color_palette)]
        plt.plot(freqs, w, label=f"tSC {k+1} (peak {pk:.1f} Hz)", color=color)
        plt.scatter([pk], [w[np.argmax(np.abs(w))]], s=40, color=color)
    
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Weight")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_mode_freq_histograms_per_component(df, col="mode_freq_hz_proj",
                                            freq_bins=np.linspace(15, 140, 20), ymax=0.25):
    comps = sorted(df["tSC_label"].unique())
    for comp in comps:
        sub = df[df["tSC_label"] == comp]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
        for j, ctype in enumerate(["phasic", "tonic"]):
            ax = axes[j]
            vals = sub.loc[sub["cycle_type"] == ctype, col].to_numpy()
            vals = vals[np.isfinite(vals)]
            if vals.size > 0:
                weights = np.ones_like(vals, dtype=float) / vals.size
                ax.hist(vals, bins=freq_bins, weights=weights, alpha=0.8)
            ax.set_xlim(15, 140); ax.set_ylim(0, ymax)
            ax.set_xlabel("Mode frequency (Hz)"); ax.set_ylabel("Probability")
            ax.set_title(f"tSC {comp} — {ctype.capitalize()}")
        plt.tight_layout(); plt.show()

def print_cycle_counts(cycles_df):
    counts = cycles_df["cycle_type"].value_counts()
    print(f"Phasic cycles: {counts.get('phasic', 0)}")
    print(f"Tonic  cycles: {counts.get('tonic', 0)}")

def plot_cycle_signature(cycle_idx, X_raw, X_z, freq_vec):
    if cycle_idx < 0 or cycle_idx >= X_raw.shape[0]:
        print(f"Invalid cycle_idx {cycle_idx}, max = {X_raw.shape[0]-1}"); return
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1); plt.plot(freq_vec, X_raw[cycle_idx], color="black")
    plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (a.u.)"); plt.title(f"Cycle {cycle_idx} — Raw signature")
    plt.subplot(1,2,2); plt.plot(freq_vec, X_z[cycle_idx], color="red")
    plt.xlabel("Frequency (Hz)"); plt.ylabel("Z-scored Power"); plt.title(f"Cycle {cycle_idx} — Z-scored signature")
    plt.tight_layout(); plt.show()

def plot_multiple_cycle_signatures(cycles_df, X_raw, X_z, freq_vec, n_cycles=10, cycle_type="phasic"):
    idxs = cycles_df.index[cycles_df["cycle_type"] == cycle_type].to_numpy()
    if len(idxs) == 0:
        print(f"No cycles of type '{cycle_type}' found."); return
    idxs = idxs[:n_cycles]
    for i, idx in enumerate(idxs, 1):
        plt.figure(figsize=(12,4))
        plt.subplot(1,2,1); plt.plot(freq_vec, X_raw[idx], color="black")
        plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (a.u.)"); plt.title(f"{cycle_type.capitalize()} cycle {i} (raw)")
        plt.subplot(1,2,2); plt.plot(freq_vec, X_z[idx], color="red")
        plt.xlabel("Frequency (Hz)"); plt.ylabel("Z-scored Power"); plt.title(f"{cycle_type.capitalize()} cycle {i} (z-scored)")
        plt.tight_layout(); plt.show()

In [ ]:
print_cycle_counts(cycles_df)
plot_tSC_weights(tSC_results["weights_freq"], tSC_results["freq_vec"])

In [ ]:
filter_ranges = [(15, 25), (25, 43), (43, 65), (65, 87), (87, 133)]

# umap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def compute_embedding_and_features(
    all_rats_data: dict,
    cycles_df: pd.DataFrame,
    mode_col: str = "mode_freq_hz_featZ_smooth",
    scale_features: bool = True,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42,
):
    """
    Build the merged dataframe (df_plot), select features (with debug prints),
    scale (optional), compute UMAP (t-SNE fallback), and return everything
    needed for plotting.
    Returns:
        emb (np.ndarray): shape (n_samples, 2)
        df_plot (pd.DataFrame): merged, filtered data aligned across sources
        color (np.ndarray): vector used for coloring (mode_col)
        xlab, ylab, method (str): axis labels and method name
        vmin, vmax (float): color scaling limits (p1, p99)
    """
    # -------- 1) Concatenate waveforms --------
    wf_frames = []
    for rid, payload in all_rats_data.items():
        wf = payload.get("waveforms_df")
        if wf is None or len(wf) == 0:
            continue
        w = wf.copy()
        if "rat_id" not in w.columns:
            w["rat_id"] = str(rid)  # ensure string
        wf_frames.append(w)

    if not wf_frames:
        raise RuntimeError("No waveforms_df found in all_rats_data.")

    waveforms_all = pd.concat(wf_frames, ignore_index=True)

    # -------- 2) Coerce key dtypes consistently --------
    def _coerce_keys(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        if "rat_id" in out.columns:
            out["rat_id"] = out["rat_id"].astype(str)
        if "cycle_type" in out.columns:
            out["cycle_type"] = out["cycle_type"].astype(str).str.lower()
        return out

    def _group_order_index(df: pd.DataFrame, group_keys):
        g = df[group_keys].copy().fillna("__NA__")
        return g.groupby(group_keys).cumcount()

    def _safe_sort(df: pd.DataFrame, prefer_order=("rat_id","cycle_type","date","SD","trial","interval_idx","cycle_idx_in_interval")):
        keys = [k for k in prefer_order if k in df.columns]
        return df.sort_values(keys, kind="mergesort") if keys else df

    waveforms_all = _coerce_keys(_safe_sort(waveforms_all))
    cycles_all    = _coerce_keys(_safe_sort(cycles_df))

    if "rat_id" not in cycles_all.columns or "cycle_type" not in cycles_all.columns:
        raise RuntimeError("cycles_df must contain 'rat_id' and 'cycle_type'.")
    if mode_col not in cycles_all.columns:
        raise RuntimeError(f"`{mode_col}` not found in cycles_df. Available: {list(cycles_all.columns)}")

    # -------- 3) Align rows via stable index within (rat_id, cycle_type) ------
    waveforms_all["gidx"] = _group_order_index(waveforms_all, ["rat_id", "cycle_type"])
    cycles_all["gidx"]    = _group_order_index(cycles_all,    ["rat_id", "cycle_type"])

    df_color = cycles_all[["rat_id","cycle_type","gidx", mode_col]].copy()
    df_plot  = waveforms_all.merge(df_color, on=["rat_id","cycle_type","gidx"], how="inner")

    if df_plot.empty:
        raise RuntimeError("No overlap between waveforms_all and cycles_df after alignment.")

    # -------- 4) Build feature matrix (INCLUDES the chosen mode_col) ----------
    numeric_cols = df_plot.select_dtypes(include=[np.number]).columns.tolist()

    meta_cols = {
        "rat_id","condition","trial","cycle_type","SD","date",
        "interval_idx","cycle_idx_in_interval","gidx"
    }
    # exclude all tSC-related numeric columns
    meta_cols |= {c for c in df_plot.columns if c.startswith("tSC")}
    # exclude all mode_* columns EXCEPT the one we want to include
    all_mode_cols = {c for c in df_plot.columns if c.startswith("mode_freq_hz_")}
    meta_cols |= (all_mode_cols - {mode_col})

    feature_cols = [c for c in numeric_cols if c not in meta_cols]

    # make sure the desired mode column is present & numeric; include it if not already
    if mode_col not in feature_cols:
        df_plot[mode_col] = pd.to_numeric(df_plot[mode_col], errors="coerce")
        feature_cols.append(mode_col)

    if not feature_cols:
        raise RuntimeError("No numeric feature columns found for embedding.")

    # ------- DEBUG PRINTS: exactly what goes into UMAP ------------------------
    print("\n=== Feature column summary ===")
    print(f"Number of features going into UMAP: {len(feature_cols)}")
    print("Feature columns:")
    for c in feature_cols:
        print("  ", c)

    print("\nPreview of the feature matrix (first 5 rows):")
    print(df_plot[feature_cols].head())

    print("\nNumeric summary (describe):")
    print(df_plot[feature_cols].describe().T)

    # -------------------------------------------------------------------------
    X = df_plot[feature_cols].to_numpy()

    from sklearn.preprocessing import StandardScaler
    if scale_features:
        X = StandardScaler().fit_transform(X)

    # color by the same mode column
    color = df_plot[mode_col].to_numpy().astype(float)

    # drop rows with non-finite entries in either X or color
    valid = np.isfinite(color) & np.isfinite(X).all(axis=1)
    df_plot = df_plot.loc[valid].reset_index(drop=True)
    X = X[valid]
    color = color[valid]

    # -------- 5) Compute embedding (UMAP→t-SNE fallback) ----------------------
    try:
        import umap
        reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                            n_components=2, random_state=random_state)
        emb = reducer.fit_transform(X)
        xlab, ylab, method = "UMAP-1", "UMAP-2", "UMAP"
    except Exception:
        from sklearn.manifold import TSNE
        tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto',
                    init='pca', random_state=random_state)
        emb = tsne.fit_transform(X)
        xlab, ylab, method = "Dim 1", "Dim 2", "t-SNE"

    # Color scaling (consistent across all plots)
    vmin = np.nanpercentile(color, 1)
    vmax = np.nanpercentile(color, 99)

    return emb, df_plot, color, xlab, ylab, method, vmin, vmax


In [ ]:
emb, df_plot, color, xlab, ylab, method, vmin, vmax = compute_embedding_and_features(
    all_rats_data=all_rats,
    cycles_df=cycles_df,
    mode_col="mode_freq_hz_featZ_smooth",
    scale_features=True,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_umap_results_split(
    emb: np.ndarray,
    df_plot: pd.DataFrame,
    color: np.ndarray,
    xlab: str,
    ylab: str,
    method: str,
    mode_col: str = "mode_freq_hz_featZ_smooth",
    vmin: float = None,
    vmax: float = None,
    small_multiples: bool = True,
    dim_x: int = 0,
    dim_y: int = 1,
):
    """
    Make split plots (TONIC and PHASIC in separate vertical subplots).
    - First: all rats -> 2 stacked subplots
    - Then (optional): per-rat -> one figure per rat, each with 2 stacked subplots
    You can choose which embedding dimensions to plot using dim_x and dim_y.
    """
    if vmin is None or vmax is None:
        vmin = np.nanpercentile(color, 1)
        vmax = np.nanpercentile(color, 99)

    # Sanity check
    n_dims = emb.shape[1]
    if dim_x >= n_dims or dim_y >= n_dims:
        raise ValueError(f"Embedding only has {n_dims} dimensions, but you asked for ({dim_x}, {dim_y})")

    # Update axis labels dynamically
    xlab_dynamic = f"{method}-{dim_x + 1}"
    ylab_dynamic = f"{method}-{dim_y + 1}"

    tonic_all = (df_plot["cycle_type"].values == "tonic")
    phasic_all = (df_plot["cycle_type"].values == "phasic")

    # -------------------- ALL RATS: split subplots --------------------
    fig, axes = plt.subplots(
        2, 1, sharex=True, sharey=True,
        figsize=(12, 10), constrained_layout=True
    )

    # TONIC (top)
    sc_tonic = axes[0].scatter(
        emb[tonic_all, dim_x], emb[tonic_all, dim_y],
        c=color[tonic_all], cmap="Blues", vmin=vmin, vmax=vmax,
        s=28, alpha=0.85, edgecolors='none'
    )
    axes[0].set_title(f"TONIC — {method} (+ {mode_col})", fontsize=12)
    axes[0].set_ylabel(ylab_dynamic, fontsize=11)

    # PHASIC (bottom)
    sc_phasic = axes[1].scatter(
        emb[phasic_all, dim_x], emb[phasic_all, dim_y],
        c=color[phasic_all], cmap="Reds", vmin=vmin, vmax=vmax,
        s=42, marker='X', edgecolors='k', linewidths=0.25
    )
    axes[1].set_title(f"PHASIC — {method} (+ {mode_col})", fontsize=12)
    axes[1].set_xlabel(xlab_dynamic, fontsize=11)
    axes[1].set_ylabel(ylab_dynamic, fontsize=11)

    # Colorbars for each subplot
    cbar_t = fig.colorbar(sc_tonic, ax=axes[0], fraction=0.046, pad=0.04)
    cbar_p = fig.colorbar(sc_phasic, ax=axes[1], fraction=0.046, pad=0.04)
    cbar_t.set_label(f"{mode_col} (Hz) — Tonic", fontsize=10)
    cbar_p.set_label(f"{mode_col} (Hz) — Phasic", fontsize=10)

    plt.show()

    # -------------------- PER-RAT: split subplots --------------------
    if small_multiples:
        rats = sorted(df_plot["rat_id"].unique())
        for rid in rats:
            m_rat = (df_plot["rat_id"].values == rid)
            tonic_m = m_rat & (df_plot["cycle_type"].values == "tonic")
            phasic_m = m_rat & (df_plot["cycle_type"].values == "phasic")

            fig, axes = plt.subplots(
                2, 1, sharex=True, sharey=True,
                figsize=(10, 8), constrained_layout=True
            )

            # TONIC (top)
            sc_tonic = axes[0].scatter(
                emb[tonic_m, dim_x], emb[tonic_m, dim_y],
                c=color[tonic_m], cmap="Blues", vmin=vmin, vmax=vmax,
                s=28, alpha=0.85, edgecolors='none'
            )
            axes[0].set_title(f"Rat {rid} — TONIC — {method} (+ {mode_col})", fontsize=12)
            axes[0].set_ylabel(ylab_dynamic, fontsize=11)

            # PHASIC (bottom)
            sc_phasic = axes[1].scatter(
                emb[phasic_m, dim_x], emb[phasic_m, dim_y],
                c=color[phasic_m], cmap="Reds", vmin=vmin, vmax=vmax,
                s=42, marker='X', edgecolors='k', linewidths=0.25
            )
            axes[1].set_title(f"Rat {rid} — PHASIC — {method} (+ {mode_col})", fontsize=12)
            axes[1].set_xlabel(xlab_dynamic, fontsize=11)
            axes[1].set_ylabel(ylab_dynamic, fontsize=11)

            # Colorbars for each subplot (same vmin/vmax)
            cbar_t = fig.colorbar(sc_tonic, ax=axes[0], fraction=0.046, pad=0.04)
            cbar_p = fig.colorbar(sc_phasic, ax=axes[1], fraction=0.046, pad=0.04)
            cbar_t.set_label(f"{mode_col} (Hz) — Tonic", fontsize=10)
            cbar_p.set_label(f"{mode_col} (Hz) — Phasic", fontsize=10)

            plt.show()

In [ ]:
print(type(emb))

In [ ]:
plot_umap_results_split(
    emb, df_plot, color, xlab, ylab, method,
    mode_col="mode_freq_hz_featZ_smooth",
    vmin=vmin, vmax=vmax,
    small_multiples=True,
)

3, 4, 7 and 8 are positive, rest are controls

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_umap_results_rat_group_split(
    emb: np.ndarray,
    df_plot: pd.DataFrame,
    color: np.ndarray,
    xlab: str,
    ylab: str,
    method: str,
    group_rats=(3, 4, 7, 8),
    rat_col: str = "rat_id",
    mode_col: str = "mode_freq_hz_featZ_smooth",
    vmin: float = None,
    vmax: float = None,
    dim_x: int = 0,
    dim_y: int = 1,
):
    """
    Make one big figure with 2 stacked subplots:
    - Top: selected rats (default 3,4,7,8)
    - Bottom: all remaining rats
    You can choose which embedding dimensions to plot with dim_x and dim_y.
    """
    if vmin is None or vmax is None:
        vmin = np.nanpercentile(color, 1)
        vmax = np.nanpercentile(color, 99)

    n_dims = emb.shape[1]
    if dim_x >= n_dims or dim_y >= n_dims:
        raise ValueError(f"Embedding only has {n_dims} dimensions, but you asked for ({dim_x}, {dim_y})")

    if rat_col not in df_plot.columns:
        raise ValueError(f"Column '{rat_col}' not found in df_plot")

    xlab_dynamic = f"{method}-{dim_x + 1}"
    ylab_dynamic = f"{method}-{dim_y + 1}"

    rat_vals = pd.to_numeric(df_plot[rat_col], errors="coerce")
    m_group = rat_vals.isin(list(group_rats)).values
    m_rest = ~m_group

    if np.sum(m_group) == 0:
        raise ValueError(f"No points found for group_rats={group_rats}")
    if np.sum(m_rest) == 0:
        raise ValueError("No points left for the 'rest' group")

    fig, axes = plt.subplots(
        2, 1, sharex=True, sharey=True,
        figsize=(13, 10), constrained_layout=True
    )

    # Group rats (top)
    sc_group = axes[0].scatter(
        emb[m_group, dim_x], emb[m_group, dim_y],
        c=color[m_group], cmap="Blues", vmin=vmin, vmax=vmax,
        s=30, alpha=0.85, edgecolors='none'
    )
    axes[0].set_title(
        f"Rats {','.join(map(str, group_rats))} — {method} (+ {mode_col}) [n={np.sum(m_group)}]",
        fontsize=12,
    )
    axes[0].set_ylabel(ylab_dynamic, fontsize=11)

    # Remaining rats (bottom)
    sc_rest = axes[1].scatter(
        emb[m_rest, dim_x], emb[m_rest, dim_y],
        c=color[m_rest], cmap="Reds", vmin=vmin, vmax=vmax,
        s=42, marker='X', edgecolors='k', linewidths=0.25
    )
    axes[1].set_title(
        f"Remaining rats — {method} (+ {mode_col}) [n={np.sum(m_rest)}]",
        fontsize=12,
    )
    axes[1].set_xlabel(xlab_dynamic, fontsize=11)
    axes[1].set_ylabel(ylab_dynamic, fontsize=11)

    cbar_g = fig.colorbar(sc_group, ax=axes[0], fraction=0.046, pad=0.04)
    cbar_r = fig.colorbar(sc_rest, ax=axes[1], fraction=0.046, pad=0.04)
    cbar_g.set_label(f"{mode_col} (Hz) — Rats {','.join(map(str, group_rats))}", fontsize=10)
    cbar_r.set_label(f"{mode_col} (Hz) — Remaining rats", fontsize=10)

    plt.show()



In [ ]:
plot_umap_results_rat_group_split(
    emb, df_plot, color, xlab, ylab, method,
    group_rats=(3, 4, 7, 8),
    mode_col="mode_freq_hz_featZ_smooth",
    vmin=vmin, vmax=vmax,
    dim_x=0, dim_y=1,
)
